## CCS 246 Final Project: Autonomous Logistics & Resource Allocation
Course: CCS 246 - Introduction to Artificial Intelligence

> Submitted by: Asoy, Bedonia, Buenavista, Fabros, Panganiban
---

### Problem Statement

Following severe storm flooding in Metro City, emergency services face heavy traffic gridlock,
flooded roads, and unpredictable road closures. The central dispatch team must replace their static
lookup system with an intelligent dispatch agent that routes ambulances and fire engines
dynamically, repositions response hubs based on historical incident clusters, navigates dynamic road
hazards, and enforces strict city safety protocols.

---

### Project Objectives

Build four foundational AI paradigms from scratch in Python to solve operational bottlenecks in autonomous logistics, routing, and resource management:

**1. Informed Pathfinding** : Finding optimal, cost-aware travel paths across spatial grids under varying terrain/traffic conditions.<br> 
**2. Local Optimization**: Solving NP-hard spatial allocation, load scheduling, or packing problems.<br> 
**3. Adversarial Decision-Making** : Navigating against dynamic disruptions, worst-case pricing spikes, or competing agents.<br> 
**4. Knowledge-Based Logic Engine**: Evaluating complex operational policies and safety rules before executing physical or resource actions.<br> 

---

### Environment Setup

This environment initializes the Metro City spatial environment using the NetworkX graph library. It constructs the 50x50 directed grid and populates the baseline state required for all four AI modules by setting baseline road traversal costs, injecting randomized traffic congestion with weights between 3.0 and 8.0, scattering prioritized incident hotspots, mapping impassable flood boundaries, and placing the initial temporary response hubs.

In [ ]:
import math
import random
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


class MetroCityEnvironment:
    """NetworkX Environment Model for Urban Emergency Dispatch & Routing.

    Integrates attributes for:
    - Module 1: Dynamic A* Search (edge weights, traversability, heuristics)
    - Module 2: Hub Location Optimization (hotspot priorities, centroid coordinates)
    - Module 3: Adversarial Navigation (hazard expansion states, dynamic blocking)
    - Module 4: Dispatch Safety Engine (depth thresholds, vehicle clearances)
    """

    def __init__(
        self,
        grid_width: int = 50,
        grid_height: int = 50,
        num_hotspots: int = 20,
        num_hubs: int = 3,
        seed: int = 42,
    ):
        np.random.seed(seed)
        random.seed(seed)

        self.width = grid_width
        self.height = grid_height
        self.num_hotspots = num_hotspots
        self.num_hubs = num_hubs

        # Instantiate NetworkX Directed Graph
        self.graph = nx.grid_2d_graph(self.width, self.height, create_using=nx.DiGraph)

        # Environment Global Attributes
        self.hubs: Dict[int, Tuple[int, int]] = {}
        self.hotspots: List[Dict] = []
        self.hazard_cells: set = set()

        # Initialize Environment Parameters
        self._initialize_node_attributes()
        self._initialize_edge_attributes()
        self.generate_hotspots()
        self.initialize_hubs()

    def _initialize_node_attributes(self) -> None:
        """Sets default node metadata across the grid."""
        for node in self.graph.nodes():
            self.graph.nodes[node]["pos"] = node
            self.graph.nodes[node]["node_type"] = "road"
            self.graph.nodes[node]["hazard_state"] = (
                "clear"  # Options: 'clear', 'flooded', 'debris'
            )
            self.graph.nodes[node]["flood_depth_m"] = 0.0  # Module 4 rule check
            self.graph.nodes[node]["hotspot_priority"] = 0  # 1 (Low) or 3 (Critical)
            self.graph.nodes[node]["is_hub"] = False

    def _initialize_edge_attributes(self) -> None:
        """Sets edge traversal costs and dynamic traversal multipliers."""
        for u, v in self.graph.edges():
            base_cost = 1.0
            congestion = 1.0
            self.graph.edges[u, v]["base_cost"] = base_cost
            self.graph.edges[u, v]["congestion_factor"] = congestion
            self.graph.edges[u, v]["weight"] = base_cost * congestion
            self.graph.edges[u, v]["is_blocked"] = False

    def generate_hotspots(self) -> None:
        """Module 2 Parameter Setup: Generates N historical incident hotspots

        with assigned emergency priority weights w_i in {1, 3}.
        """
        self.hotspots.clear()
        selected_nodes = random.sample(list(self.graph.nodes()), self.num_hotspots)

        for idx, node in enumerate(selected_nodes):
            priority = random.choice([1, 3])
            self.graph.nodes[node]["node_type"] = "hotspot"
            self.graph.nodes[node]["hotspot_priority"] = priority

            self.hotspots.append(
                {
                    "id": idx,
                    "coords": node,
                    "priority": priority,  # w_i multiplier for Module 2 optimization
                }
            )

    def initialize_hubs(self) -> None:
        """Module 2 Parameter Setup: Places K response hubs randomly initially."""
        self.hubs.clear()
        hub_nodes = random.sample(list(self.graph.nodes()), self.num_hubs)

        for k, node in enumerate(hub_nodes):
            self.hubs[k] = node
            self.graph.nodes[node]["is_hub"] = True

    def inject_traffic_congestion(
        self, congested_ratio: float = 0.15, min_cost: float = 3.0, max_cost: float = 8.0
    ) -> None:
        """Module 1 Parameter Setup: Assigns congestion costs [3.0, 8.0] to edges."""
        edges = list(self.graph.edges())
        num_congested = int(len(edges) * congested_ratio)
        congested_edges = random.sample(edges, num_congested)

        for u, v in congested_edges:
            congestion = np.random.uniform(min_cost, max_cost)
            self.graph.edges[u, v]["congestion_factor"] = congestion
            self.graph.edges[u, v]["weight"] = (
                self.graph.edges[u, v]["base_cost"] * congestion
            )

    def update_hazard_state(
        self, cell: Tuple[int, int], flood_depth: float = 0.5
    ) -> None:
        """Module 3 & 4 Parameter Update: Blocks cell and connected edges due to dynamic hazard."""
        if cell in self.graph:
            self.graph.nodes[cell]["hazard_state"] = "flooded"
            self.graph.nodes[cell]["flood_depth_m"] = flood_depth
            self.hazard_cells.add(cell)

            for u, v in list(self.graph.in_edges(cell)) + list(
                self.graph.out_edges(cell)
            ):
                self.graph.edges[u, v]["is_blocked"] = True
                self.graph.edges[u, v]["weight"] = float("inf")

    def get_manhattan_heuristic(
        self, node_a: Tuple[int, int], node_b: Tuple[int, int]
    ) -> float:
        """Module 1 Heuristic Function h_1(n): Computes Manhattan Distance."""
        return float(abs(node_a[0] - node_b[0]) + abs(node_a[1] - node_b[1]))

    def visualize_environment(self, title: str = "Metro City Graph Environment") -> None:
        """Plots the NetworkX grid with node states, hubs, hotspots, and hazards."""
        fig, ax = plt.subplots(figsize=(10, 10))
        pos = {node: node for node in self.graph.nodes()}

        road_nodes = [
            n
            for n, d in self.graph.nodes(data=True)
            if d["node_type"] == "road" and d["hazard_state"] == "clear"
        ]
        hazard_nodes = list(self.hazard_cells)
        low_pri_hotspots = [
            n for n in self.graph.nodes() if self.graph.nodes[n]["hotspot_priority"] == 1
        ]
        crit_pri_hotspots = [
            n for n in self.graph.nodes() if self.graph.nodes[n]["hotspot_priority"] == 3
        ]
        hub_nodes = [coords for coords in self.hubs.values()]

        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=road_nodes,
            node_size=10,
            node_color="lightgrey",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=hazard_nodes,
            node_size=40,
            node_color="black",
            node_shape="s",
            label="Flooded/Hazard",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=low_pri_hotspots,
            node_size=50,
            node_color="blue",
            label="Hotspot (Priority 1)",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=crit_pri_hotspots,
            node_size=90,
            node_color="red",
            label="Hotspot (Priority 3)",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=hub_nodes,
            node_size=150,
            node_color="green",
            node_shape="^",
            label="Response Hub (K)",
            ax=ax,
        )

        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xlabel("X Grid Coordinate")
        ax.set_ylabel("Y Grid Coordinate")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right")
        plt.tight_layout()
        plt.show()


# =============================================================================
# Execution Test
# =============================================================================
if __name__ == "__main__":
    env = MetroCityEnvironment(
        grid_width=50, grid_height=50, num_hotspots=25, num_hubs=4, seed=101
    )
    print("Environment successfully initialized without errors.")
    print(f"Nodes: {env.graph.number_of_nodes()}, Edges: {env.graph.number_of_edges()}")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def render_metro_city_graph(
    env: MetroCityEnvironment, title: str = "Metro City Topology & Resource Allocation"
) -> None:
    """Extracts node attributes from the MetroCityEnvironment instance and renders

    a spatial visualization of the urban grid environment.

    Visual Key:
    - Grey dots: Clear, navigable road nodes
    - Black squares: Flooded / impassable hazard cells (Module 3 & 4)
    - Blue dots: Priority 1 incident hotspots (Module 2)
    - Red dots: Priority 3 critical incident hotspots (Module 2)
    - Green triangles: K temporary response hubs (Module 2)
    """
    fig, ax = plt.subplots(figsize=(12, 12))

    # Node positions correspond directly to grid coordinates (x, y)
    pos = {node: node for node in env.graph.nodes()}

    # Categorize nodes based on attribute state
    road_nodes = [
        n
        for n, d in env.graph.nodes(data=True)
        if d["node_type"] == "road" and d["hazard_state"] == "clear"
    ]
    hazard_nodes = list(env.hazard_cells)
    low_pri_hotspots = [
        n for n in env.graph.nodes() if env.graph.nodes[n]["hotspot_priority"] == 1
    ]
    crit_pri_hotspots = [
        n for n in env.graph.nodes() if env.graph.nodes[n]["hotspot_priority"] == 3
    ]
    hub_nodes = [coords for coords in env.hubs.values()]

    # Render Node Layers
    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=road_nodes,
        node_size=12,
        node_color="#d3d3d3",
        alpha=0.5,
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=hazard_nodes,
        node_size=45,
        node_color="black",
        node_shape="s",
        label="Flooded / Hazard (Impassable)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=low_pri_hotspots,
        node_size=65,
        node_color="#1f77b4",
        label="Hotspot (Priority 1: Standard)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=crit_pri_hotspots,
        node_size=110,
        node_color="#d62728",
        label="Hotspot (Priority 3: Critical)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=hub_nodes,
        node_size=180,
        node_color="#2ca02c",
        node_shape="^",
        edgecolors="black",
        linewidths=1.5,
        label="Response Hub ($K$)",
        ax=ax,
    )

    # Configure Axes and Grid Details
    ax.set_title(title, fontsize=15, fontweight="bold", pad=15)
    ax.set_xlabel("X Grid Coordinate", fontsize=12)
    ax.set_ylabel("Y Grid Coordinate", fontsize=12)
    ax.set_xlim(-1, env.width)
    ax.set_ylim(-1, env.height)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)

    plt.tight_layout()
    plt.show()


# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    # 1. Initialize environment model instance
    city_env = MetroCityEnvironment(
        grid_width=50, grid_height=50, num_hotspots=25, num_hubs=4, seed=101
    )

    # 2. Inject sample hazard data (simulating dynamic flood boundary)
    for x in range(20, 26):
        for y in range(20, 26):
            city_env.update_hazard_state((x, y), flood_depth=0.8)

    # 3. Render the network graph visualization
    render_metro_city_graph(
        city_env,
        title="Metro City NetworkX Topology (Grid: 50x50, N=25 Hotspots, K=4 Hubs)",
    )

---
### Module 1 — Dynamic Route Planning & Heuristic Analysis

In [ ]:
# Write Code Here

--- 
### Module 2 — Response Hub Location Optimization

Places K = 4 temporary hubs to minimize the priority-weighted average Manhattan delay to the N = 25 hotspots (weight 1 = standard, 3 = critical).

**Method:** weighted k-medians. Repeat **assign** (each hotspot to its nearest hub) and **update** (each hub moves to the weighted median of its hotspots) until no hub moves. Restarts keep the best local optimum, and hubs are snapped off flooded cells.

In [ ]:
import random
from typing import List, Tuple

HubPoint = Tuple[int, int]

def hub_manhattan(a: HubPoint, b: HubPoint) -> int:
    """Grid distance between two cells."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def hub_weighted_median(values: List[int], weights: List[float]) -> int:
    """Return the value where half of the total weight lies on each side."""
    pairs = sorted(zip(values, weights))
    half = sum(weights) / 2
    running = 0.0
    for value, weight in pairs:
        running += weight
        if running >= half:
            return value
    return pairs[-1][0]  # safety fallback, normally never reached

def hub_assign(points: List[HubPoint], hubs: List[HubPoint]) -> List[int]:
    """For each point, return the index of its nearest hub."""
    return [
        min(range(len(hubs)), key=lambda h: hub_manhattan(p, hubs[h]))
        for p in points
    ]

def hub_weighted_cost(points, weights, hubs) -> float:
    """Average priority-weighted Manhattan delay to the nearest hub."""
    total = sum(
        w * min(hub_manhattan(p, h) for h in hubs) for p, w in zip(points, weights)
    )
    return total / sum(weights)


def hub_solve(points, weights, k, seed=0, max_iter=100):
    """One weighted k-medians run. Returns (hubs, labels, cost)."""
    rng = random.Random(seed)
    hubs = rng.sample(points, k)  # start on k distinct hotspots
    for _ in range(max_iter):
        labels = hub_assign(points, hubs)
        new_hubs = []
        for h in range(k):
            members = [i for i, lab in enumerate(labels) if lab == h]
            if not members:  # empty cluster: keep that hub where it is
                new_hubs.append(hubs[h])
                continue
            xs = [points[i][0] for i in members]
            ys = [points[i][1] for i in members]
            ws = [weights[i] for i in members]
            new_hubs.append((hub_weighted_median(xs, ws), hub_weighted_median(ys, ws)))
        if new_hubs == hubs:  # nothing moved, so converged
            break
        hubs = new_hubs
    labels = hub_assign(points, hubs)
    return hubs, labels, hub_weighted_cost(points, weights, hubs)


def hub_self_test() -> None:
    """Check the 3-point worked example: the hub must land on the heavy point."""
    hubs, _, cost = hub_solve([(2, 2), (6, 2), (6, 10)], [1, 1, 3], k=1)
    assert hubs == [(6, 10)] and cost == 4.0
    print("Self-test passed.")

hub_self_test()

In [ ]:
def hub_snap_to_free_cell(cell, blocked, width, height, taken=()):
    """Nearest grid cell that is not flooded and not already used by a hub."""
    best = None
    for x in range(width):
        for y in range(height):
            if (x, y) in blocked or (x, y) in taken:
                continue
            d = hub_manhattan(cell, (x, y))
            if best is None or d < best[0]:
                best = (d, (x, y))
    return best[1]


def hub_best_of_restarts(points, weights, k, blocked, width, height, restarts=20):
    """Run hub_solve with several seeds, snap to safe cells, keep the cheapest.

    Returns (hubs, labels, cost, all_costs)
    """
    best = None
    all_costs = []
    for seed in range(restarts):
        hubs, _, _ = hub_solve(points, weights, k, seed=seed)
        safe = []
        for h in hubs:
            safe.append(hub_snap_to_free_cell(h, blocked, width, height, taken=safe))
        cost = hub_weighted_cost(points, weights, safe)
        all_costs.append(cost)
        if best is None or cost < best[2]:
            best = (safe, hub_assign(points, safe), cost)
    return best[0], best[1], best[2], all_costs


def hub_snap_self_test() -> None:
    """Check snapping: flooded cell moves out, free cell stays, taken cell is avoided"""
    flood = {(x, y) for x in range(20, 26) for y in range(20, 26)}
    assert hub_snap_to_free_cell((22, 22), flood, 50, 50) == (19, 22)
    assert hub_snap_to_free_cell((5, 5), flood, 50, 50) == (5, 5)
    assert hub_snap_to_free_cell((5, 5), flood, 50, 50, taken=[(5, 5)]) == (4, 5)
    print("Snap self-test passed.")

hub_snap_self_test()

In [ ]:
K_HUBS = 4
env = city_env  
hs_points = [h["coords"] for h in env.hotspots]
hs_weights = [h["priority"] for h in env.hotspots]
initial_hubs = list(env.hubs.values())

def hub_validate(env, hubs, k):
    """Raise AssertionError if the hub layout breaks a basic rule."""
    assert len(hubs) == k, "wrong number of hubs"
    assert len(set(hubs)) == k, "two hubs share a cell"
    for x, y in hubs:
        assert 0 <= x < env.width and 0 <= y < env.height, "hub off the grid"
        assert (x, y) not in env.hazard_cells, "hub inside flood"
    print("All hub checks passed.")


def hub_apply(env, hubs):
    """Write the final hubs back into the shared environment."""
    for node in env.hubs.values():
        env.graph.nodes[node]["is_hub"] = False
    env.hubs = {k: h for k, h in enumerate(hubs)}
    for h in hubs:
        env.graph.nodes[h]["is_hub"] = True


final_hubs, final_labels, final_cost, run_costs = hub_best_of_restarts(
    hs_points, hs_weights, K_HUBS, env.hazard_cells, env.width, env.height
)
hub_validate(env, final_hubs, K_HUBS)
hub_apply(env, final_hubs)
print("Initial hubs :", initial_hubs)
print("Final hubs   :", final_hubs)
print("Weighted avg delay before:", round(hub_weighted_cost(hs_points, hs_weights, initial_hubs), 2))
print("Weighted avg delay after :", round(final_cost, 2))

---
### Module 3 — Tactical Adversarial Hazard Navigation

In [ ]:
# Write Code Here

---
### Module 4 — Automated Dispatch Safety Verification Engine

In [ ]:
# Write Code Here

---
### End-to-End Simulation

In [ ]:
# Write Code Here